#### 文本流输出

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool

model = init_chat_model('deepseek-chat')

In [2]:
full = None
for chunk in model.stream('what color is the sky.'):
    full = chunk if not full else full + chunk
    print(full.text)


The
The color
The color of
The color of the
The color of the sky
The color of the sky is
The color of the sky is most
The color of the sky is most commonly
The color of the sky is most commonly **
The color of the sky is most commonly **blue
The color of the sky is most commonly **blue**
The color of the sky is most commonly **blue** during
The color of the sky is most commonly **blue** during the
The color of the sky is most commonly **blue** during the day
The color of the sky is most commonly **blue** during the day.


The color of the sky is most commonly **blue** during the day.

This
The color of the sky is most commonly **blue** during the day.

This happens
The color of the sky is most commonly **blue** during the day.

This happens because
The color of the sky is most commonly **blue** during the day.

This happens because of
The color of the sky is most commonly **blue** during the day.

This happens because of **
The color of the sky is most commonly **blue** during the day

In [3]:
type(full)

langchain_core.messages.ai.AIMessageChunk

In [4]:
full.content_blocks

[{'type': 'text',
  'text': 'The color of the sky is most commonly **blue** during the day.\n\nThis happens because of **Rayleigh scattering** — sunlight interacts with molecules and tiny particles in Earth\'s atmosphere, scattering shorter (blue) wavelengths of light more than longer (red/orange) wavelengths. So, blue light is scattered across the sky, making it appear blue to our eyes.\n\nHowever, the sky can appear different colors depending on conditions:\n\n- **Sunrise/Sunset:** Red, orange, pink, or purple — because sunlight passes through more atmosphere, scattering more blue light away and leaving longer wavelengths.\n- **Overcast:** Gray or white — because clouds scatter all wavelengths of light equally.\n- **Night:** Black — because there’s no sunlight to scatter (except for starlight, moonlight, and artificial lights).\n- **During storms or unusual atmospheric conditions:** Greenish, yellow, or other colors (rare).\n\nSo while we usually say "the sky is blue," its color is d

In [5]:
full.text

'The color of the sky is most commonly **blue** during the day.\n\nThis happens because of **Rayleigh scattering** — sunlight interacts with molecules and tiny particles in Earth\'s atmosphere, scattering shorter (blue) wavelengths of light more than longer (red/orange) wavelengths. So, blue light is scattered across the sky, making it appear blue to our eyes.\n\nHowever, the sky can appear different colors depending on conditions:\n\n- **Sunrise/Sunset:** Red, orange, pink, or purple — because sunlight passes through more atmosphere, scattering more blue light away and leaving longer wavelengths.\n- **Overcast:** Gray or white — because clouds scatter all wavelengths of light equally.\n- **Night:** Black — because there’s no sunlight to scatter (except for starlight, moonlight, and artificial lights).\n- **During storms or unusual atmospheric conditions:** Greenish, yellow, or other colors (rare).\n\nSo while we usually say "the sky is blue," its color is dynamic and changes with time

#### 工具调用流输出

In [6]:
reasoner = init_chat_model(model='deepseek-reasoner')

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city"""
    return f"It's always sunny in {city}."

reasoner = reasoner.bind_tools([get_weather_for_location])

In [11]:
full = None
for chunk in reasoner.stream('what is the weather in sf?'):
    for block in chunk.content_blocks:
        if block['type'] == 'reasoning' and (reasoning := block.get('reasoning')):
            print(reasoning, end='')
        elif block['type'] == 'tool_call_chunk':
            print(block)
        elif block['type'] == 'text':
            print(block['text'], end='')
    full = chunk if not full else full + chunk

The user is asking about the weather in "sf". I need to interpret what "sf" means. It could be San Francisco. I should use the get_weather_for_location tool with city parameter set to "San Francisco". Let me do that.{'type': 'tool_call_chunk', 'id': 'call_00_brEKGnGmHbDxcHMc2LdlBHeA', 'name': 'get_weather_for_location', 'args': '', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '{', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'city', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ': ', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'San', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ' Francisco', 'index': 0}
{'type'

In [15]:
type(full)

langchain_core.messages.ai.AIMessageChunk

In [12]:
full.content_blocks

[{'type': 'reasoning',
  'reasoning': 'The user is asking about the weather in "sf". I need to interpret what "sf" means. It could be San Francisco. I should use the get_weather_for_location tool with city parameter set to "San Francisco". Let me do that.'},
 {'type': 'tool_call',
  'id': 'call_00_brEKGnGmHbDxcHMc2LdlBHeA',
  'name': 'get_weather_for_location',
  'args': {'city': 'San Francisco'}}]

In [13]:
full.text

''

In [14]:
full.tool_calls

[{'name': 'get_weather_for_location',
  'args': {'city': 'San Francisco'},
  'id': 'call_00_brEKGnGmHbDxcHMc2LdlBHeA',
  'type': 'tool_call'}]